# Doctor Registration Testing Notebook

This notebook provides a comprehensive guide to test the MeroDaktar doctor registration system.

## Features:
- Doctor registration with full profile
- Login and authentication
- Profile management
- Schedule setup
- Appointment handling

## Setup & Configuration

Import required libraries and set up the API endpoint.

In [ ]:
import requests
import json
from datetime import datetime, timedelta
from pprint import pprint

# API Configuration
BASE_URL = "http://localhost:8000/api/v1"
HEADERS = {"Content-Type": "application/json"}

# Store tokens and IDs for later use
doctor_token = None
doctor_id = None

print("✅ Setup complete!")
print(f"API Base URL: {BASE_URL}")

## 1. Doctor Registration

Register a new doctor with complete profile information.

In [ ]:
# Doctor Registration Data
doctor_data = {
    "email": f"dr.smith.{datetime.now().strftime('%Y%m%d%H%M%S')}@hospital.com",
    "password": "SecurePassword123!",
    "full_name": "Dr. Sarah Smith",
    "specialization": "Cardiology",
    "license_number": f"MED{datetime.now().strftime('%Y%m%d%H%M')}",
    "phone": "+977-9841234567",
    "qualifications": "MBBS, MD (Cardiology), FACC",
    "experience_years": 12,
    "consultation_fee": 1500.00,
    "bio": "Experienced cardiologist specializing in interventional cardiology and heart failure management.",
    "hospital_affiliation": "Grande International Hospital",
    "address": "Kathmandu, Nepal"
}

print("📝 Registering new doctor...")
print(f"Email: {doctor_data['email']}")
print(f"Name: {doctor_data['full_name']}")
print(f"Specialization: {doctor_data['specialization']}\n")

# Send registration request
response = requests.post(
    f"{BASE_URL}/auth/register/doctor",
    json=doctor_data,
    headers=HEADERS
)

if response.status_code == 201:
    result = response.json()
    doctor_id = result.get('id')
    print("✅ Doctor registered successfully!")
    print(f"\nDoctor ID: {doctor_id}")
    print(f"Email: {result.get('email')}")
    print(f"Name: {result.get('full_name')}")
    print(f"Verification Status: {'Verified' if result.get('is_verified') else 'Pending Verification'}")
    print("\n📋 Full Response:")
    pprint(result)
else:
    print(f"❌ Registration failed!")
    print(f"Status Code: {response.status_code}")
    print(f"Error: {response.text}")

## 2. Doctor Login

Login with the registered doctor credentials to get an access token.

In [ ]:
# Login credentials
login_data = {
    "username": doctor_data['email'],
    "password": doctor_data['password']
}

print("🔐 Logging in...")
print(f"Email: {login_data['username']}\n")

# Send login request (form-urlencoded)
response = requests.post(
    f"{BASE_URL}/auth/login/doctor",
    data=login_data,
    headers={"Content-Type": "application/x-www-form-urlencoded"}
)

if response.status_code == 200:
    result = response.json()
    doctor_token = result.get('access_token')
    print("✅ Login successful!")
    print(f"\nAccess Token: {doctor_token[:50]}...")
    print(f"Token Type: {result.get('token_type')}")
    print(f"\n📋 Full Response:")
    pprint(result)
else:
    print(f"❌ Login failed!")
    print(f"Status Code: {response.status_code}")
    print(f"Error: {response.text}")

## 3. Get Doctor Profile

Retrieve the doctor's profile information using the access token.

In [ ]:
if not doctor_token:
    print("❌ No token available. Please login first (run cell 2).")
else:
    print("👤 Fetching doctor profile...\n")
    
    # Send request with authorization header
    response = requests.get(
        f"{BASE_URL}/doctors/me",
        headers={"Authorization": f"Bearer {doctor_token}"}
    )
    
    if response.status_code == 200:
        profile = response.json()
        print("✅ Profile retrieved successfully!\n")
        print(f"Name: {profile.get('full_name')}")
        print(f"Specialization: {profile.get('specialization')}")
        print(f"License: {profile.get('license_number')}")
        print(f"Experience: {profile.get('experience_years')} years")
        print(f"Consultation Fee: NPR {profile.get('consultation_fee')}")
        print(f"Hospital: {profile.get('hospital_affiliation')}")
        print(f"Phone: {profile.get('phone')}")
        print(f"Email: {profile.get('email')}")
        print(f"Verified: {'Yes' if profile.get('is_verified') else 'No'}")
        print(f"\n📋 Full Profile:")
        pprint(profile)
    else:
        print(f"❌ Failed to fetch profile!")
        print(f"Status Code: {response.status_code}")
        print(f"Error: {response.text}")

## 4. Update Doctor Profile

Update specific fields in the doctor's profile.

In [ ]:
if not doctor_token:
    print("❌ No token available. Please login first (run cell 2).")
else:
    # Updated data
    update_data = {
        "bio": "Senior cardiologist with 12+ years of experience in interventional procedures, heart failure management, and preventive cardiology. Special interest in cardiac imaging and non-invasive diagnostics.",
        "consultation_fee": 2000.00,
        "qualifications": "MBBS, MD (Cardiology), FACC, Fellowship in Interventional Cardiology"
    }
    
    print("✏️ Updating doctor profile...\n")
    
    response = requests.put(
        f"{BASE_URL}/doctors/me",
        json=update_data,
        headers={
            "Authorization": f"Bearer {doctor_token}",
            "Content-Type": "application/json"
        }
    )
    
    if response.status_code == 200:
        profile = response.json()
        print("✅ Profile updated successfully!\n")
        print(f"New Consultation Fee: NPR {profile.get('consultation_fee')}")
        print(f"Updated Qualifications: {profile.get('qualifications')}")
        print(f"\nUpdated Bio:\n{profile.get('bio')}")
        print(f"\n📋 Updated Profile:")
        pprint(profile)
    else:
        print(f"❌ Update failed!")
        print(f"Status Code: {response.status_code}")
        print(f"Error: {response.text}")

## 5. Set Up Doctor Schedule

Create availability schedule for different days of the week.

In [ ]:
if not doctor_token:
    print("❌ No token available. Please login first (run cell 2).")
else:
    # Define schedule for multiple days
    schedules = [
        {
            "day_of_week": 1,  # Monday
            "start_time": "09:00",
            "end_time": "17:00",
            "slot_duration_minutes": 30,
            "is_available": True
        },
        {
            "day_of_week": 2,  # Tuesday
            "start_time": "09:00",
            "end_time": "17:00",
            "slot_duration_minutes": 30,
            "is_available": True
        },
        {
            "day_of_week": 3,  # Wednesday
            "start_time": "10:00",
            "end_time": "16:00",
            "slot_duration_minutes": 30,
            "is_available": True
        },
        {
            "day_of_week": 4,  # Thursday
            "start_time": "09:00",
            "end_time": "17:00",
            "slot_duration_minutes": 30,
            "is_available": True
        },
        {
            "day_of_week": 5,  # Friday
            "start_time": "09:00",
            "end_time": "15:00",
            "slot_duration_minutes": 30,
            "is_available": True
        }
    ]
    
    days_map = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday", 5: "Friday", 6: "Saturday", 0: "Sunday"}
    
    print("📅 Setting up doctor schedule...\n")
    
    created_schedules = []
    
    for schedule in schedules:
        response = requests.post(
            f"{BASE_URL}/schedules/my-schedule",
            json=schedule,
            headers={
                "Authorization": f"Bearer {doctor_token}",
                "Content-Type": "application/json"
            }
        )
        
        if response.status_code == 201:
            result = response.json()
            created_schedules.append(result)
            day_name = days_map.get(schedule['day_of_week'], 'Unknown')
            print(f"✅ {day_name}: {schedule['start_time']} - {schedule['end_time']} (30 min slots)")
        else:
            day_name = days_map.get(schedule['day_of_week'], 'Unknown')
            print(f"❌ {day_name}: Failed - {response.text}")
    
    print(f"\n✅ Created {len(created_schedules)} schedule entries!")
    print("\n📋 Schedule Summary:")
    pprint(created_schedules)

## 6. View Doctor Schedule

Retrieve and display the doctor's weekly schedule.

In [ ]:
if not doctor_token:
    print("❌ No token available. Please login first (run cell 2).")
else:
    print("📅 Fetching doctor schedule...\n")
    
    response = requests.get(
        f"{BASE_URL}/schedules/my-schedule",
        headers={"Authorization": f"Bearer {doctor_token}"}
    )
    
    if response.status_code == 200:
        schedules = response.json()
        days_map = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday", 5: "Friday", 6: "Saturday", 0: "Sunday"}
        
        print("✅ Schedule retrieved successfully!\n")
        print("=" * 60)
        print(f"{'Day':<15} {'Time':<20} {'Duration':<15} {'Status':<10}")
        print("=" * 60)
        
        for schedule in sorted(schedules, key=lambda x: x['day_of_week']):
            day = days_map.get(schedule['day_of_week'], 'Unknown')
            time_range = f"{schedule['start_time']} - {schedule['end_time']}"
            duration = f"{schedule['slot_duration_minutes']} mins"
            status = "Available" if schedule['is_available'] else "Unavailable"
            print(f"{day:<15} {time_range:<20} {duration:<15} {status:<10}")
        
        print("=" * 60)
        print(f"\nTotal schedule entries: {len(schedules)}")
        print("\n📋 Full Schedule Data:")
        pprint(schedules)
    else:
        print(f"❌ Failed to fetch schedule!")
        print(f"Status Code: {response.status_code}")
        print(f"Error: {response.text}")

## 7. Get Doctor Statistics

View the doctor's dashboard statistics including appointments and patients.

In [ ]:
if not doctor_token:
    print("❌ No token available. Please login first (run cell 2).")
else:
    print("📊 Fetching doctor statistics...\n")
    
    response = requests.get(
        f"{BASE_URL}/dashboard/doctor/stats",
        headers={"Authorization": f"Bearer {doctor_token}"}
    )
    
    if response.status_code == 200:
        stats = response.json()
        print("✅ Statistics retrieved successfully!\n")
        print("=" * 50)
        print("📈 DOCTOR DASHBOARD STATISTICS")
        print("=" * 50)
        print(f"Total Appointments:      {stats.get('total_appointments', 0)}")
        print(f"Today's Appointments:    {stats.get('today_appointments', 0)}")
        print(f"Pending Appointments:    {stats.get('pending_appointments', 0)}")
        print(f"Completed Appointments:  {stats.get('completed_appointments', 0)}")
        print(f"Total Patients:          {stats.get('total_patients', 0)}")
        print("=" * 50)
        print("\n📋 Full Statistics:")
        pprint(stats)
    else:
        print(f"❌ Failed to fetch statistics!")
        print(f"Status Code: {response.status_code}")
        print(f"Error: {response.text}")

## 8. View Doctor Appointments

Get all appointments for the doctor.

In [ ]:
if not doctor_token:
    print("❌ No token available. Please login first (run cell 2).")
else:
    print("📋 Fetching doctor appointments...\n")
    
    response = requests.get(
        f"{BASE_URL}/appointments/doctor/appointments",
        headers={"Authorization": f"Bearer {doctor_token}"}
    )
    
    if response.status_code == 200:
        appointments = response.json()
        print(f"✅ Found {len(appointments)} appointment(s)!\n")
        
        if appointments:
            print("=" * 80)
            for i, apt in enumerate(appointments, 1):
                print(f"\n📅 Appointment #{i}")
                print(f"   Patient: {apt.get('patient_name', 'Unknown')}")
                print(f"   Date: {apt.get('appointment_date')}")
                print(f"   Time: {apt.get('appointment_time')}")
                print(f"   Reason: {apt.get('reason', 'Not specified')}")
                print(f"   Status: {apt.get('status', 'pending').upper()}")
                print(f"   Type: {apt.get('appointment_type', 'general')}")
            print("\n" + "=" * 80)
            print("\n📋 Full Appointments Data:")
            pprint(appointments)
        else:
            print("No appointments found yet.")
    else:
        print(f"❌ Failed to fetch appointments!")
        print(f"Status Code: {response.status_code}")
        print(f"Error: {response.text}")

## 9. Search for Doctors (Public Endpoint)

Search for registered doctors by specialization or name.

In [ ]:
# Search parameters
search_params = {
    "specialization": "Cardiology",
    "skip": 0,
    "limit": 10
}

print(f"🔍 Searching for doctors...")
print(f"Specialization: {search_params['specialization']}\n")

response = requests.get(
    f"{BASE_URL}/doctors/search",
    params=search_params
)

if response.status_code == 200:
    doctors = response.json()
    print(f"✅ Found {len(doctors)} doctor(s)!\n")
    
    if doctors:
        print("=" * 80)
        for i, doc in enumerate(doctors, 1):
            print(f"\n👨‍⚕️ Doctor #{i}")
            print(f"   Name: {doc.get('full_name')}")
            print(f"   Specialization: {doc.get('specialization')}")
            print(f"   Experience: {doc.get('experience_years')} years")
            print(f"   Consultation Fee: NPR {doc.get('consultation_fee')}")
            print(f"   Hospital: {doc.get('hospital_affiliation', 'Not specified')}")
            print(f"   Verified: {'Yes' if doc.get('is_verified') else 'No'}")
            if doc.get('bio'):
                print(f"   Bio: {doc.get('bio')[:100]}...")
        print("\n" + "=" * 80)
        print("\n📋 Full Search Results:")
        pprint(doctors)
    else:
        print("No doctors found matching the criteria.")
else:
    print(f"❌ Search failed!")
    print(f"Status Code: {response.status_code}")
    print(f"Error: {response.text}")

## 10. Summary

Display a summary of the registered doctor account.

In [ ]:
print("=" * 70)
print("📊 DOCTOR REGISTRATION SUMMARY")
print("=" * 70)

if doctor_id and doctor_token:
    print(f"\n✅ Doctor Account Successfully Created!\n")
    print(f"Doctor ID:     {doctor_id}")
    print(f"Email:         {doctor_data.get('email')}")
    print(f"Name:          {doctor_data.get('full_name')}")
    print(f"Specialization: {doctor_data.get('specialization')}")
    print(f"License:       {doctor_data.get('license_number')}")
    print(f"\n🔑 Authentication Token: {doctor_token[:30]}...")
    print(f"\n💡 You can now use this account to:")
    print(f"   • Manage appointments")
    print(f"   • View patient encounters")
    print(f"   • Update schedule")
    print(f"   • Add clinical notes")
    print(f"   • Generate reports")
else:
    print("\n⚠️ No doctor account created yet.")
    print("Please run the registration and login cells first.")

print("\n" + "=" * 70)
print("\n📝 Next Steps:")
print("   1. Patients can now book appointments with this doctor")
print("   2. Doctor can manage appointments from the dashboard")
print("   3. Doctor can view patient medical history")
print("   4. Doctor can add clinical notes and prescriptions")
print("=" * 70)